# CLIP-ReID: Обработка видео

```
video.mp4
   │
   ▼  YOLOv8 — детекция людей
 crops [N, 3, 256, 128]
   │
   ▼  ONNX / TensorRT — CLIP-ReID эмбеддинги
 embeddings [N, 1280]
   │
   ▼  cosine matching — присвоение ID
 data/
   ├── json/persons_000001.json   (по 100 персон)
   └── persons/
         ├── 1/video_00-00-01-680.png
         └── 2/...
```

In [ ]:
# !pip install ultralytics opencv-python onnxruntime-gpu
# !pip install tensorrt pycuda   # только для TRT бэкенда

import os, sys, json, gc, time
import numpy as np
import torch
import torchvision.transforms as T
import cv2
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

REPO_DIR = os.path.dirname(os.path.abspath("__file__"))
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"torch   : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

## Настройка

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                   НАСТРОЙТЕ ПОД ВАШЕ ОКРУЖЕНИЕ              ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Пути к видео (список) ─────────────────────────────────────────────────
VIDEO_PATHS = [
    "videos/camera1.mp4",
    # "videos/camera2.mp4",
]

# ── ReID бэкенд ───────────────────────────────────────────────────────────
BACKEND      = "onnx"              # "onnx" или "trt"
ONNX_PATH    = "clipreid_msmt17.onnx"
TRT_PATH     = "clipreid_msmt17.trt"
REID_BATCH   = 32                  # сколько кропов за один прогон модели

# ── Детектор YOLOv8 ───────────────────────────────────────────────────────
YOLO_MODEL   = "yolov8m.pt"        # n / s / m / l / x — точность vs скорость
YOLO_CONF    = 0.50                # минимальная уверенность детекции
YOLO_IOU     = 0.40                # NMS IoU порог
MIN_CROP_PX  = 32                  # отбрасываем кропы < 32px по меньшей стороне

# ── Трекинг ───────────────────────────────────────────────────────────────
SIM_THRESHOLD = 0.65               # cosine similarity порог «тот же человек»

# ── Обработка видео ───────────────────────────────────────────────────────
SKIP_FRAMES  = 1                   # брать каждый N-й кадр (1 = все)
MAX_FRAMES   = None                # None = до конца видео

# ── Выходные данные ───────────────────────────────────────────────────────
DATA_DIR          = "data"
SAVE_CROPS        = True
PERSONS_PER_JSON  = 100

# ── Производительность ────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Backend  : {BACKEND.upper()}")
print(f"Device   : {DEVICE}")
print(f"Data dir : {os.path.abspath(DATA_DIR)}")

## ReID бэкенды (ONNX / TRT)

In [ ]:
# CLIP нормализация
TRANSFORM = T.Compose([
    T.Resize((256, 128), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(
        mean=[0.48145466, 0.4578275,  0.40821073],
        std= [0.26862954, 0.26130258, 0.27577711],
    )
])


class ONNXModel:
    """ONNX Runtime инференс (CPU или CUDA)."""

    def __init__(self, onnx_path: str):
        import onnxruntime as ort
        providers = (
            ["CUDAExecutionProvider", "CPUExecutionProvider"]
            if torch.cuda.is_available() else ["CPUExecutionProvider"]
        )
        opts = ort.SessionOptions()
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.session  = ort.InferenceSession(onnx_path, opts, providers=providers)
        self.provider = self.session.get_providers()[0]

        out_shape = self.session.get_outputs()[0].shape
        self.emb_dim = out_shape[-1] if out_shape[-1] and out_shape[-1] > 0 else 1280
        print(f"  ONNX провайдер : {self.provider}")
        print(f"  EMB DIM        : {self.emb_dim}")

    def infer(self, imgs_np: np.ndarray) -> np.ndarray:
        """imgs_np: [B, 3, 256, 128] float32 → [B, EMB_DIM] float32"""
        return self.session.run(["embedding"], {"image": imgs_np})[0]

    def close(self):
        del self.session
        gc.collect()


class TRTModel:
    """TensorRT инференс."""

    def __init__(self, trt_path: str):
        import tensorrt as trt
        import pycuda.driver as cuda
        import pycuda.autoinit
        self._trt, self._cuda = trt, cuda

        logger  = trt.Logger(trt.Logger.WARNING)
        runtime = trt.Runtime(logger)
        with open(trt_path, "rb") as f:
            self.engine  = runtime.deserialize_cuda_engine(f.read())
        self.context = self.engine.create_execution_context()
        self.stream  = cuda.Stream()
        self._alloc_buffers()

        self.emb_dim = self._out["max_shape"][-1]
        print(f"  TRT engine загружен")
        print(f"  EMB DIM : {self.emb_dim}")

    def _alloc_buffers(self):
        self._inp = self._out = None
        for i in range(self.engine.num_io_tensors):
            name  = self.engine.get_tensor_name(i)
            dtype = self._trt.nptype(self.engine.get_tensor_dtype(name))
            mode  = self.engine.get_tensor_mode(name)
            ms    = self.engine.get_tensor_profile_shape(name, 0)[2]
            h     = self._cuda.pagelocked_empty(int(np.prod(ms)), dtype)
            d     = self._cuda.mem_alloc(h.nbytes)
            entry = {"name": name, "host": h, "device": d, "max_shape": ms}
            if mode == self._trt.TensorIOMode.INPUT:
                self._inp = entry
            else:
                self._out = entry

    def infer(self, imgs_np: np.ndarray) -> np.ndarray:
        b = imgs_np.shape[0]
        self.context.set_input_shape("image", imgs_np.shape)
        np.copyto(self._inp["host"][:imgs_np.size], imgs_np.ravel())
        self._cuda.memcpy_htod_async(self._inp["device"], self._inp["host"], self.stream)
        self.context.execute_async_v3(stream_handle=self.stream.handle)
        self._cuda.memcpy_dtoh_async(self._out["host"], self._out["device"], self.stream)
        self.stream.synchronize()
        return self._out["host"][:b * self.emb_dim].reshape(b, self.emb_dim).copy()

    def close(self):
        del self.context, self.engine
        for buf in [self._inp, self._out]:
            buf["device"].free()
        gc.collect()


def load_reid_model(backend: str):
    print(f"Загружаем ReID модель ({backend.upper()})...")
    if backend == "onnx":
        return ONNXModel(ONNX_PATH)
    elif backend == "trt":
        return TRTModel(TRT_PATH)
    raise ValueError(f"Неизвестный бэкенд: {backend}")


def preprocess_crops(crops_pil: list) -> np.ndarray:
    """Список PIL изображений → [N, 3, 256, 128] float32"""
    return torch.stack([TRANSFORM(c) for c in crops_pil]).numpy()


def normalize_embeddings(embs: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    return embs / np.clip(norms, 1e-8, None)

## Детектор (YOLOv8)

In [ ]:
class PersonDetector:
    """
    Детектор людей на основе YOLOv8.

    Возвращает только bbox с class_id=0 (person).
    Фильтрует слишком маленькие кропы (< MIN_CROP_PX).
    """

    def __init__(self, model_name: str, conf: float, iou: float, device: str):
        from ultralytics import YOLO
        self.model  = YOLO(model_name)
        self.conf   = conf
        self.iou    = iou
        self.device = device
        print(f"  YOLO model : {model_name}")
        print(f"  conf={conf}  iou={iou}  device={device}")

    def detect(self, frame_bgr: np.ndarray) -> list:
        """
        Детектирует людей на кадре.

        Returns:
            list of (bbox_xyxy, conf)
            bbox_xyxy = [x1, y1, x2, y2] в пикселях
        """
        results = self.model(
            frame_bgr,
            conf=self.conf,
            iou=self.iou,
            classes=[0],          # только person
            device=self.device,
            verbose=False,
        )

        detections = []
        h, w = frame_bgr.shape[:2]

        for r in results:
            if r.boxes is None or len(r.boxes) == 0:
                continue
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf = float(box.conf[0])

                # Клиппинг к размеру кадра
                x1 = max(0, int(x1));  y1 = max(0, int(y1))
                x2 = min(w, int(x2));  y2 = min(h, int(y2))

                # Фильтр слишком маленьких боксов
                if (x2 - x1) < MIN_CROP_PX or (y2 - y1) < MIN_CROP_PX:
                    continue

                detections.append(([x1, y1, x2, y2], conf))

        return detections

## Галерея персон — трекинг по эмбеддингам

In [ ]:
class PersonGallery:
    """
    Онлайн-галерея персон для ReID.

    Для каждой детекции:
      1. Вычисляет cosine similarity с усреднёнными эмбеддингами галереи
      2. Если max_sim >= threshold → та же персона (обновляет эмбеддинг)
      3. Иначе → новая персона, новый ID

    Усреднение онлайн:
      new_mean = (old_mean * n + new_emb) / (n + 1)  → потом L2-нормализация
    """

    def __init__(self, threshold: float = 0.65):
        self.threshold = threshold
        # pid → {'emb': np[D], 'count': int, 'appearances': list}
        self.persons: dict = {}
        self._next_id = 1
        # Кэш матрицы эмбеддингов для быстрого поиска [N, D]
        self._emb_matrix: np.ndarray = None
        self._ids: list = []
        self._dirty = False          # True = нужно пересчитать кэш

    # ── Публичный API ──────────────────────────────────────────────────────

    def process_frame_detections(
        self,
        embeddings:   np.ndarray,    # [N, D] — уже L2-нормализованы
        bboxes:       list,          # [N] → [x1, y1, x2, y2]
        confs:        list,          # [N] float
        video_name:   str,
        frame_number: int,
        timestamp:    str,
    ) -> list:
        """
        Обрабатывает все детекции с одного кадра.
        Возвращает list of (pid, bbox) для сохранения кропов.
        """
        assigned = []
        used_pids = set()  # чтобы одной персоне не назначить два бокса

        for emb, bbox, conf in zip(embeddings, bboxes, confs):
            pid = self._match(emb, exclude=used_pids)
            used_pids.add(pid)

            # Обновляем эмбеддинг (онлайн среднее)
            self._update_emb(pid, emb)

            # Сохраняем визит
            x1, y1, x2, y2 = bbox
            appearance = {
                "video":         video_name,
                "bboxes":        [x1, x2, y1, y2],  # формат: x1, x2, y1, y2
                "conf":          round(float(conf), 4),
                "n_frames":      frame_number,
                "time_in_video": timestamp,
            }
            self.persons[pid]["appearances"].append(appearance)
            assigned.append((pid, bbox))

        return assigned

    @property
    def num_persons(self) -> int:
        return len(self.persons)

    # ── Внутренние методы ─────────────────────────────────────────────────

    def _match(self, emb: np.ndarray, exclude: set) -> int:
        if not self.persons:
            return self._new_person(emb)

        if self._dirty:
            self._rebuild_cache()

        sims = self._emb_matrix @ emb            # [N]

        # Маскируем уже занятые персоны в этом кадре
        for pid in exclude:
            if pid in self._ids:
                sims[self._ids.index(pid)] = -1.0

        best_idx = int(np.argmax(sims))
        best_sim = float(sims[best_idx])

        if best_sim >= self.threshold:
            return self._ids[best_idx]
        return self._new_person(emb)

    def _new_person(self, emb: np.ndarray) -> int:
        pid = self._next_id
        self.persons[pid] = {"emb": emb.copy(), "count": 1, "appearances": []}
        self._next_id += 1
        self._dirty = True
        return pid

    def _update_emb(self, pid: int, emb: np.ndarray):
        p = self.persons[pid]
        n = p["count"]
        updated = (p["emb"] * n + emb) / (n + 1)
        norm = np.linalg.norm(updated)
        p["emb"] = updated / (norm + 1e-8)
        p["count"] = n + 1
        self._dirty = True

    def _rebuild_cache(self):
        self._ids = list(self.persons.keys())
        self._emb_matrix = np.stack(
            [self.persons[pid]["emb"] for pid in self._ids]
        )  # [N, D]
        self._dirty = False

## Менеджер данных — сохранение JSON и кропов

In [ ]:
def format_timestamp(seconds: float) -> str:
    """
    Секунды → HH:MM:SS.mmm
    Пример: 91.68 → '00:01:31.680'
    """
    ms  = int((seconds % 1) * 1000)
    s   = int(seconds) % 60
    m   = int(seconds) // 60 % 60
    h   = int(seconds) // 3600
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"


def timestamp_to_filename(ts: str) -> str:
    """'00:01:31.680' → '00-01-31-680' (безопасно для файловой системы)"""
    return ts.replace(":", "-").replace(".", "-")


class DataManager:
    """
    Управляет сохранением данных:

    data/
    ├── json/
    │   ├── persons_000001.json   ← IDs 1–100
    │   ├── persons_000101.json   ← IDs 101–200
    │   └── ...
    └── persons/
          ├── 1/
          │   ├── camera1_00-00-01-680.png
          │   └── camera1_00-00-05-120.png
          ├── 2/
          └── ...

    JSON формат (один элемент):
    {
      "id": 1,
      "embeddings": [1280 floats],
      "appearances": [
        {
          "video": "camera1",
          "bboxes": [x1, x2, y1, y2],
          "conf": 0.95,
          "n_frames": 42,
          "time_in_video": "00:00:01.680"
        }
      ]
    }
    """

    def __init__(self, data_dir: str, persons_per_file: int = 100,
                 save_crops: bool = True):
        self.data_dir         = data_dir
        self.persons_per_file = persons_per_file
        self.save_crops       = save_crops

        self.json_dir    = os.path.join(data_dir, "json")
        self.persons_dir = os.path.join(data_dir, "persons")
        os.makedirs(self.json_dir,    exist_ok=True)
        os.makedirs(self.persons_dir, exist_ok=True)

    def save_crop(
        self,
        pid:        int,
        crop_bgr:   np.ndarray,
        video_name: str,
        timestamp:  str,
    ) -> str:
        """
        Сохраняет кроп персоны.
        Путь: data/persons/{pid}/{video_name}_{HH-MM-SS-mmm}.png
        """
        if not self.save_crops or crop_bgr is None or crop_bgr.size == 0:
            return ""
        person_dir = os.path.join(self.persons_dir, str(pid))
        os.makedirs(person_dir, exist_ok=True)
        ts_safe = timestamp_to_filename(timestamp)
        fname   = f"{video_name}_{ts_safe}.png"
        path    = os.path.join(person_dir, fname)
        cv2.imwrite(path, crop_bgr)
        return path

    def flush_gallery_to_json(self, gallery: PersonGallery, tag: str = "") -> list:
        """
        Сериализует всю галерею в JSON файлы по PERSONS_PER_FILE персон.

        Returns:
            list of saved file paths
        """
        persons_sorted = sorted(gallery.persons.items(), key=lambda x: x[0])
        saved_files = []

        for batch_start in range(0, len(persons_sorted), self.persons_per_file):
            batch = persons_sorted[batch_start : batch_start + self.persons_per_file]
            start_id = batch[0][0]

            data = [
                {
                    "id":          pid,
                    "embeddings":  [round(float(v), 6) for v in info["emb"]],
                    "appearances": info["appearances"],
                }
                for pid, info in batch
            ]

            prefix = f"{tag}_" if tag else ""
            fname  = f"{prefix}persons_{start_id:06d}.json"
            fpath  = os.path.join(self.json_dir, fname)

            with open(fpath, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)

            saved_files.append(fpath)
            print(f"  Сохранено: {fpath}  ({len(batch)} персон)")

        return saved_files

## Обработка видео

In [ ]:
def process_video(
    video_path:  str,
    reid_model,
    detector:    PersonDetector,
    gallery:     PersonGallery,
    data_mgr:    DataManager,
    batch_size:  int  = 32,
    skip_frames: int  = 1,
    max_frames:  int  = None,
) -> dict:
    """
    Обрабатывает одно видео:
    1. Читает кадры (cv2)
    2. Детектирует людей (YOLO)
    3. Батчем извлекает эмбеддинги (CLIP-ReID)
    4. Сопоставляет с галереей
    5. Сохраняет кропы

    JSON сохраняется отдельно вызовом data_mgr.flush_gallery_to_json()

    Returns:
        dict со статистикой обработки
    """
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    cap = cv2.VideoCapture(video_path)
    assert cap.isOpened(), f"Не удалось открыть видео: {video_path}"

    fps        = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total      = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    limit      = min(total, max_frames) if max_frames else total
    n_process  = (limit + skip_frames - 1) // skip_frames

    print(f"\n{'─'*55}")
    print(f"  Видео     : {video_name}")
    print(f"  Разрешение: {width}×{height}  FPS={fps:.1f}")
    print(f"  Кадров    : {total}  (обрабатываем каждый {skip_frames}-й → {n_process})")
    print(f"{'─'*55}")

    stats = {
        "video": video_name, "frames_total": total,
        "frames_processed": 0, "detections": 0, "new_persons": 0,
    }

    frame_num = 0
    pbar = tqdm(total=n_process, desc=f"  {video_name}", unit="frame")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_num += 1

        if max_frames and frame_num > max_frames:
            break
        if frame_num % skip_frames != 0:
            continue

        timestamp = format_timestamp(frame_num / fps)

        # 1. Детекция
        detections = detector.detect(frame)
        if not detections:
            pbar.update(1)
            stats["frames_processed"] += 1
            continue

        # 2. Формируем батч кропов
        crops_pil  = []
        crops_bgr  = []
        bboxes     = []
        confs      = []

        for bbox, conf in detections:
            x1, y1, x2, y2 = bbox
            crop_bgr = frame[y1:y2, x1:x2]
            crop_pil = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
            crops_pil.append(crop_pil)
            crops_bgr.append(crop_bgr)
            bboxes.append(bbox)
            confs.append(conf)

        # 3. Батчевый инференс ReID (по batch_size кропов за раз)
        all_embs = []
        for i in range(0, len(crops_pil), batch_size):
            batch_np = preprocess_crops(crops_pil[i : i + batch_size])
            embs     = reid_model.infer(batch_np)
            all_embs.append(embs)
        all_embs = normalize_embeddings(np.concatenate(all_embs, axis=0))

        # 4. Сопоставление с галереей
        persons_before = gallery.num_persons
        assigned = gallery.process_frame_detections(
            embeddings=all_embs,
            bboxes=bboxes,
            confs=confs,
            video_name=video_name,
            frame_number=frame_num,
            timestamp=timestamp,
        )

        # 5. Сохраняем кропы
        for (pid, bbox), crop_bgr in zip(assigned, crops_bgr):
            data_mgr.save_crop(pid, crop_bgr, video_name, timestamp)

        stats["detections"]   += len(detections)
        stats["new_persons"]  += gallery.num_persons - persons_before
        stats["frames_processed"] += 1

        pbar.set_postfix({
            "det": stats["detections"],
            "ids": gallery.num_persons,
        })
        pbar.update(1)

    pbar.close()
    cap.release()

    print(f"  Кадров обработано  : {stats['frames_processed']}")
    print(f"  Детекций всего     : {stats['detections']}")
    print(f"  Новых персон       : {stats['new_persons']}")
    print(f"  Всего персон в БД  : {gallery.num_persons}")
    return stats

## Запуск

In [ ]:
# ── Инициализация ──────────────────────────────────────────────────────────
reid_model = load_reid_model(BACKEND)

print("\nЗагружаем детектор...")
detector = PersonDetector(
    model_name=YOLO_MODEL,
    conf=YOLO_CONF,
    iou=YOLO_IOU,
    device=DEVICE,
)

gallery  = PersonGallery(threshold=SIM_THRESHOLD)
data_mgr = DataManager(
    data_dir=DATA_DIR,
    persons_per_file=PERSONS_PER_JSON,
    save_crops=SAVE_CROPS,
)

print(f"\nДиректория данных: {os.path.abspath(DATA_DIR)}")
print("Инициализация завершена ✓")

In [ ]:
# ── Обработка всех видео ───────────────────────────────────────────────────
all_stats = []
t_start   = time.time()

for video_path in VIDEO_PATHS:
    if not os.path.exists(video_path):
        print(f"⚠ Видео не найдено: {video_path}")
        continue

    stats = process_video(
        video_path=video_path,
        reid_model=reid_model,
        detector=detector,
        gallery=gallery,
        data_mgr=data_mgr,
        batch_size=REID_BATCH,
        skip_frames=SKIP_FRAMES,
        max_frames=MAX_FRAMES,
    )
    all_stats.append(stats)

total_elapsed = time.time() - t_start
print(f"\nВсего видео обработано : {len(all_stats)}")
print(f"Всего персон в галерее : {gallery.num_persons}")
print(f"Полное время           : {total_elapsed:.1f} сек")

In [ ]:
# ── Сохранение JSON ────────────────────────────────────────────────────────
print(f"Сохраняем {gallery.num_persons} персон в JSON...")
saved = data_mgr.flush_gallery_to_json(gallery)

print(f"\nИтог:")
print(f"  JSON файлов : {len(saved)}")
print(f"  Персон      : {gallery.num_persons}")
if SAVE_CROPS:
    n_crops = sum(
        len(os.listdir(os.path.join(DATA_DIR, "persons", d)))
        for d in os.listdir(os.path.join(DATA_DIR, "persons"))
        if os.path.isdir(os.path.join(DATA_DIR, "persons", d))
    )
    print(f"  Кропов сохранено: {n_crops}")

# Освобождаем ресурсы
reid_model.close()
print("\nГотово ✓")

## Проверка результатов

In [ ]:
# Читаем первый JSON и показываем структуру
import glob

json_files = sorted(glob.glob(os.path.join(DATA_DIR, "json", "*.json")))
print(f"JSON файлов: {len(json_files)}")

if json_files:
    with open(json_files[0], encoding="utf-8") as f:
        sample = json.load(f)

    print(f"\nПример персоны из {os.path.basename(json_files[0])}:")
    p = sample[0]
    print(f"  id          : {p['id']}")
    print(f"  embeddings  : len={len(p['embeddings'])}  min={min(p['embeddings']):.4f}  max={max(p['embeddings']):.4f}")
    print(f"  appearances : {len(p['appearances'])} визитов")
    if p['appearances']:
        a = p['appearances'][0]
        print(f"  \nПример визита:")
        for k, v in a.items():
            print(f"    {k:15s}: {v}")

    print(f"\nСтруктура папки persons/:")
    persons_root = os.path.join(DATA_DIR, "persons")
    for d in sorted(os.listdir(persons_root))[:5]:
        n = len(os.listdir(os.path.join(persons_root, d)))
        print(f"  {d}/  ({n} кропов)")